In [1]:
import pandas as pd

hour_cols = [f'Hour {i}' for i in range(1, 25)]

dtype_map = {col: str for col in ['Delivery Date', 'Generator', 'Fuel Type', 'Measurement']}
dtype_map.update({col: float for col in hour_cols})


df_all = pd.read_csv("../data/2023/PUB_GenOutputCapabilityMonth_202301.csv",
    dtype=dtype_map,
    parse_dates=['Delivery Date'],
    na_values=[' ']
)

# df_all.columns.tolist()


In [ ]:
# Filter to Output measurement only
output_df = df_all[df_all['Measurement'] == 'Output'].copy()

hour_cols = [f'Hour {i}' for i in range(1, 25)]

# Summary stats per generator
summary = output_df.groupby('Generator')[hour_cols].apply(
    lambda x: pd.Series({
        'mean_output': x.values.mean(),
        'std_output': x.values.std(),
        'max_output': x.values.max(),
        'zero_pct': (x.values == 0).mean(),
        'full_pct': (x.values == x.values.max()).mean()
    })
).sort_values('mean_output', ascending=False)

# print(summary.to_string())

                       mean_output  std_output  max_output  zero_pct  full_pct
Generator                                                                     
RAILBEDWF-LT.AG_SR      150.611979   91.941319       262.0  0.005208  0.015625
K2WIND                  134.205729   99.107170       264.0  0.059896  0.039062
ARMOW                    87.190104   64.947237       178.0  0.054688  0.023438
WEST LINCOLN NRWF        84.932292   74.554130       221.0  0.028646  0.002604
HENVEY SOUTH             77.684896   59.074831       176.0  0.062500  0.002604
COMBER                   71.690104   49.558418       149.0  0.026042  0.018229
UNDERWOOD                71.091146   59.696276       175.0  0.125000  0.007812
JERICHO                  69.937500   46.832773       130.0  0.054688  0.010417
AMARANTH                 63.833333   51.413645       162.0  0.065104  0.002604
GRANDWF                  62.940104   51.851190       144.0  0.070312  0.028646
PORTALMA-T3              54.851562   33.705315      

In [5]:
import pandas as pd
from pathlib import Path

def analyze_folder(folder: str):
    hour_cols = [f'Hour {i}' for i in range(1, 25)]
    dtype_map = {col: str for col in ['Delivery Date', 'Generator', 'Fuel Type', 'Measurement']}
    dtype_map.update({col: float for col in hour_cols})

    anomalies = []

    for csv_path in sorted(Path(folder).glob('*.csv')):
        df = pd.read_csv(csv_path, dtype=dtype_map, parse_dates=['Delivery Date'], na_values=[' '])

        wind = df[df['Fuel Type'] == 'WIND']
        output = wind[wind['Measurement'] == 'Output']
        capacity = wind[wind['Measurement'] == 'Available Capacity']

        for gen in output['Generator'].unique():
            gen_output = output.loc[output['Generator'] == gen, hour_cols].values
            gen_cap = capacity.loc[capacity['Generator'] == gen, hour_cols].values

            mean_out = gen_output.mean()
            mean_cap = gen_cap.mean() if gen_cap.size > 0 else 0

            if mean_out == 0:
                anomalies.append({
                    'file': csv_path.name,
                    'generator': gen,
                    'issue': 'always_zero',
                    'mean_output': 0,
                    'mean_capacity': mean_cap,
                    'capacity_factor': 0,
                })
            elif mean_cap > 0 and mean_out < 0.15 * mean_cap:
                anomalies.append({
                    'file': csv_path.name,
                    'generator': gen,
                    'issue': 'low_output',
                    'mean_output': round(mean_out, 2),
                    'mean_capacity': round(mean_cap, 2),
                    'capacity_factor': round(mean_out / mean_cap, 4),
                })

    return pd.DataFrame(anomalies)

anomalies_df = analyze_folder('../data/2026')
print(anomalies_df.to_string(index=False))

                                   file             generator       issue  mean_output  mean_capacity  capacity_factor
PUB_GenOutputCapabilityMonth_202602.csv MCLEANSMTNWF-LT.AG_T1 always_zero         0.00          60.00           0.0000
PUB_GenOutputCapabilityMonth_202603.csv MCLEANSMTNWF-LT.AG_T1  low_output         3.69          59.30           0.0623
PUB_GenOutputCapabilityMonth_202604.csv         CEDAR POINT 2  low_output         0.02           4.58           0.0042


In [9]:
import pandas as pd
from pathlib import Path

hour_cols = [f'Hour {i}' for i in range(1, 25)]
dtype_map = {col: str for col in ['Delivery Date', 'Generator', 'Fuel Type', 'Measurement']}
dtype_map.update({col: float for col in hour_cols})

nan_records = []

for csv_path in sorted(Path('../data/2026').glob('*.csv')):
    df = pd.read_csv(csv_path, dtype=dtype_map, parse_dates=['Delivery Date'], na_values=[' '])
    wind = df[df['Fuel Type'] == 'WIND']

    for measurement in ['Output', 'Available Capacity']:
        subset = wind[wind['Measurement'] == measurement]
        nan_count = subset[hour_cols].isna().sum().sum()
        total_cells = subset[hour_cols].size

        if nan_count > 0:
            nan_records.append({
                'file': csv_path.name,
                'measurement': measurement,
                'nan_count': nan_count,
                'total_cells': total_cells,
                'nan_pct': round(nan_count / total_cells * 100, 2),
            })

nan_df = pd.DataFrame(nan_records)
print(nan_df.to_string(index=False) if len(nan_df) > 0 else 'No NaN values found')

                                   file measurement  nan_count  total_cells  nan_pct
PUB_GenOutputCapabilityMonth_202601.csv      Output         32        33480     0.10
PUB_GenOutputCapabilityMonth_202602.csv      Output          3        30240     0.01
PUB_GenOutputCapabilityMonth_202603.csv      Output         66        33480     0.20
PUB_GenOutputCapabilityMonth_202604.csv      Output         47        32400     0.15


In [ ]:
# interpolation for NaN values
# output_df[hour_cols] = output_df.groupby('Generator')[hour_cols].transform(lambda x: x.interpolate(method='linear', axis=0, limit_direction='both'))

,Delivery Date,Generator,Fuel Type,Measurement,Hour 1,Hour 2,Hour 3,Hour 4,Hour 5,Hour 6,...,Hour 15,Hour 16,Hour 17,Hour 18,Hour 19,Hour 20,Hour 21,Hour 22,Hour 23,Hour 24
0,2026-01-01,ABKENORA,HYDRO,Capability,11.0,11.0,11.0,11.0,11.0,11.0,...,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0
1,2026-01-01,ABKENORA,HYDRO,Output,12.0,12.0,12.0,12.0,12.0,12.0,...,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0
2,2026-01-01,ADELAIDE,WIND,Available Capacity,60.0,60.0,60.0,60.0,60.0,60.0,...,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0
3,2026-01-01,ADELAIDE,WIND,Forecast,59.0,58.0,58.0,58.0,58.0,57.0,...,32.0,32.0,17.0,9.0,11.0,17.0,23.0,34.0,29.0,31.0
4,2026-01-01,ADELAIDE,WIND,Output,59.0,59.0,59.0,59.0,59.0,57.0,...,32.0,32.0,16.0,8.0,12.0,18.0,25.0,36.0,28.0,33.0


: 

In [20]:
df = df_all[df_all['Fuel Type'] == 'WIND']

print(df['Generator'].nunique())
print(sorted(df['Generator'].unique()))

45
['ADELAIDE', 'AMARANTH', 'AMHERST ISLAND', 'ARMOW', 'BELLE RIVER', 'BLAKE', 'BORNISH', 'BOW LAKE', 'BOW LAKE 2', 'CEDAR POINT 2', 'COMBER', 'CRYSLER', 'DILLON', 'EAST LAKE', 'ERIEAU', 'GOSFIELDWGS', 'GOSHEN', 'GOULAIS', 'GRAND VALLEY 3', 'GRANDWF', 'GREENWICH', 'HENVEY NORTH', 'HENVEY SOUTH', 'JERICHO', 'K2WIND', 'KINGSBRIDGE', 'LANDON', 'MCLEANSMTNWF-LT.AG_T1', 'NORTH KENT', 'PAROCHES', 'PORT BURWELL', 'PORTALMA-T1', 'PORTALMA-T3', 'PRINCEFARM', 'RAILBEDWF-LT.AG_SR', 'RIPLEY SOUTH', 'ROMNEY', 'SANDUSK-LT.AG_T1', 'SHANNON', 'SPENCE', 'SUMMERHAVEN', 'UNDERWOOD', 'WEST LINCOLN NRWF', 'WOLFE ISLAND', 'ZURICH']


In [12]:
df.head()

,Delivery Date,Generator,Fuel Type,Measurement,Hour 1,Hour 2,Hour 3,Hour 4,Hour 5,Hour 6,...,Hour 15,Hour 16,Hour 17,Hour 18,Hour 19,Hour 20,Hour 21,Hour 22,Hour 23,Hour 24
2,2026-01-01,ADELAIDE,WIND,Available Capacity,60.0,60.0,60.0,60.0,60.0,60.0,...,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0
3,2026-01-01,ADELAIDE,WIND,Forecast,59.0,58.0,58.0,58.0,58.0,57.0,...,32.0,32.0,17.0,9.0,11.0,17.0,23.0,34.0,29.0,31.0
4,2026-01-01,ADELAIDE,WIND,Output,59.0,59.0,59.0,59.0,59.0,57.0,...,32.0,32.0,16.0,8.0,12.0,18.0,25.0,36.0,28.0,33.0
9,2026-01-01,AMARANTH,WIND,Available Capacity,153.0,153.0,153.0,124.0,124.0,124.0,...,124.0,124.0,124.0,124.0,124.0,124.0,124.0,124.0,124.0,124.0
10,2026-01-01,AMARANTH,WIND,Forecast,71.0,59.0,75.0,56.0,36.0,37.0,...,77.0,71.0,60.0,51.0,65.0,62.0,61.0,49.0,45.0,38.0


In [15]:
dtype_map = {col: float for col in ['Total Project Capacity (MW)', 'Rotor Diameter (m)', 'Hub Height (m)', 'Latitude', 'Longitude']}
# dtype_map.update({col: float for col in hour_cols})


df_cwtd = pd.read_csv("../data/Wind_Turbine_Database_FGP.csv",
    dtype=dtype_map,
    parse_dates=['Commissioning'],
    na_values=[' ']
)

df_cwtd.columns.tolist()

['Province_Territoire',
 'Province_Territoire.1',
 'Project Name',
 'Total Project Capacity (MW)',
 'Turbine Identifier',
 'Turbine Number',
 'Number of Turbines in Project',
 'Turbine Rated Capacity (kW)',
 'Rotor Diameter (m)',
 'Hub Height (m)',
 'Manufacturer',
 'Model',
 'Commissioning',
 'Latitude',
 'Longitude',
 'Notes']

In [16]:
df_cwtd.head()

,Province_Territoire,Province_Territoire.1,Project Name,Total Project Capacity (MW),Turbine Identifier,Turbine Number,Number of Turbines in Project,Turbine Rated Capacity (kW),Rotor Diameter (m),Hub Height (m),Manufacturer,Model,Commissioning,Latitude,Longitude,Notes
0,Ontario,Ontario,OPG 7 Gomberg,0.0000,OPG1,1,1,0,80.00,78.0,Vestas,V80-1.8,2001,43.812073,-79.073921,"Turbine decommissioned in 2019, original capac..."
1,Ontario,Ontario,Living Labs,0.0104,LVL2,2,3,2.4,3.72,15.2,Southwest Windpower,Skystream 3.7,2012,43.831305,-79.588094,NaN
2,Ontario,Ontario,Living Labs,0.0104,LVL1,1,3,1,2.50,17.4,Bergey,Excel 1-48,2012,43.831699,-79.589118,NaN
3,Ontario,Ontario,Living Labs,0.0104,LVL3,3,3,7,1.93,18.3,Darrieus-Savonius,HiVAWT,2015,43.831306,-79.588087,Vertical Axis Hybrid
4,Ontario,Ontario,Hog-Tied Farms,0.2500,HTF1,1,1,250,30.00,30.0,Wind Energy Solutions,WES 250,2007,43.104582,-81.857571,"250 kW, two-bladed model"


In [22]:
print(df_cwtd['Project Name'].nunique())
print(sorted(df_cwtd['Project Name'].unique()))

100
['Adelaide Wind Energy Centre', 'Amherst Island Wind Project', 'Armow Wind Project', 'Arthur', 'Belle River', 'Bisnett Line (Thames River I)', 'Bluewater Wind Farm', 'Bornish Wind Energy Centre', 'Bow Lake Wind Project', 'CAW Wind Turbine', 'Cedar Point', 'Chatham', 'Clear Creek', 'Comber Wind Farm', 'Conestogo', 'Cruickshank', 'Cultus', 'Dufferin Wind', 'East Durham', 'East Lake St. Clair Wind', 'Erie Shores', 'Erieau Wind', 'Ernestown', 'Exhibition Place Turbine', 'Ferndale', 'Frogmore', 'Front Line (Thames River I)', 'Ganaraska', 'Gesner', 'Gosfield', 'Goshen Wind Energy', 'Goulais Wind Farm', 'Gracey (Thames River II)', 'Grand Bend Wind Farm', 'Grand Renewable Wind', 'Grand Valley', 'Greenwich Renewable Energy Project', 'Grey Highlands Clean Energy', 'Grey Highlands Zero Emissions People', "Gunn's Hill", 'HAF Energy', 'Harrow', 'Henvey Inlet Wind Farm', 'Hog-Tied Farms', 'Huron Wind', 'Jericho', 'K2 Wind Power Facility', 'Kent Breeze Wind Farm', 'Kingsbridge I Wind Power', 'Liv

In [58]:
k2_forecast.head()

,Delivery Date,Generator,Fuel Type,Measurement,Hour 1,Hour 2,Hour 3,Hour 4,Hour 5,Hour 6,...,Hour 15,Hour 16,Hour 17,Hour 18,Hour 19,Hour 20,Hour 21,Hour 22,Hour 23,Hour 24
208,2026-01-01,K2WIND,WIND,Forecast,244.0,247.0,247.0,237.0,244.0,239.0,...,233.0,244.0,246.0,247.0,247.0,248.0,248.0,248.0,247.0,245.0
634,2026-01-02,K2WIND,WIND,Forecast,242.0,243.0,237.0,216.0,227.0,227.0,...,194.0,196.0,147.0,150.0,196.0,199.0,196.0,170.0,117.0,124.0
1060,2026-01-03,K2WIND,WIND,Forecast,157.0,169.0,173.0,148.0,174.0,208.0,...,58.0,58.0,49.0,56.0,84.0,99.0,129.0,133.0,140.0,108.0
1486,2026-01-04,K2WIND,WIND,Forecast,104.0,101.0,127.0,121.0,120.0,130.0,...,146.0,94.0,51.0,22.0,5.0,0.0,0.0,6.0,16.0,54.0
1912,2026-01-05,K2WIND,WIND,Forecast,140.0,124.0,162.0,186.0,237.0,235.0,...,9.0,1.0,0.0,0.0,0.0,0.0,1.0,5.0,12.0,22.0


In [ ]:
k2_output['Delivery Date'].min(), k2_output['Delivery Date'].max()

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-01-31 00:00:00'))

In [59]:
# Melt all measurements together
k2_all = k2.melt(
    id_vars=['Delivery Date', 'Generator', 'Fuel Type', 'Measurement'],
    value_vars=hour_cols,
    var_name='Hour',
    value_name='MWh'
)

k2_all['Hour'] = k2_all['Hour'].str.extract(r'(\d+)').astype(int)

# Pivot so each measurement is a column
k2_pivot = k2_all.pivot_table(
    index=['Delivery Date', 'Hour'],
    columns='Measurement',
    values='MWh'
).reset_index()

k2_pivot.columns.name = None
k2_pivot.head()

,Delivery Date,Hour,Available Capacity,Forecast,Output
0,2026-01-01,1,270.0,244.0,247.0
1,2026-01-01,2,270.0,247.0,248.0
2,2026-01-01,3,270.0,247.0,246.0
3,2026-01-01,4,270.0,237.0,239.0
4,2026-01-01,5,270.0,244.0,244.0


In [67]:
k2_pivot['forecast_gap'] = k2_pivot['Forecast'] - k2_pivot['Output']

k2_pivot['gap_percent'] = k2_pivot['forecast_gap'] / k2_pivot['Forecast'] * 100

# Summary stats
k2_pivot['gap_percent'].describe()



count    733.000000
mean       0.889553
std       15.429829
min     -200.000000
25%       -1.195219
50%        0.000000
75%        0.649351
max      100.000000
Name: gap_percent, dtype: float64

In [68]:
# Hours where output was significantly below forecast (possible curtailment)
threshold = 10  # MW, adjust based on what you see
k2_pivot[k2_pivot['gap_percent'].abs() > threshold].sort_values('forecast_gap', ascending=False)

,Delivery Date,Hour,Available Capacity,Forecast,Output,forecast_gap,gap_percent
295,2026-01-13,8,270.0,178.0,133.0,45.0,25.280899
293,2026-01-13,6,270.0,99.0,66.0,33.0,33.333333
426,2026-01-18,19,270.0,162.0,139.0,23.0,14.197531
679,2026-01-29,8,270.0,75.0,66.0,9.0,12.000000
214,2026-01-09,23,270.0,65.0,57.0,8.0,12.307692
...,...,...,...,...,...,...,...
360,2026-01-16,1,270.0,42.0,50.0,-8.0,-19.047619
95,2026-01-04,24,270.0,54.0,63.0,-9.0,-16.666667
708,2026-01-30,13,270.0,44.0,54.0,-10.0,-22.727273
614,2026-01-26,15,270.0,36.0,49.0,-13.0,-36.111111
